In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/creditcard.csv")

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
#0-normal, 1- fraud
X = df.drop("Class", axis=1)
y = df["Class"]

print(X.shape)
print(y.value_counts())

(284807, 30)
Class
0    284315
1       492
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Train fraud count:", y_train.sum())
print("Test fraud count:", y_test.sum())

Train size: (227845, 30)
Test size: (56962, 30)
Train fraud count: 394
Test fraud count: 98


Scale for Logistic Regression

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test_scaled[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

### Logistic regression Model 
Baseline

In [6]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_model.fit(X_train_scaled, y_train)

log_probs = log_model.predict_proba(X_test_scaled)[:, 1]
log_preds = (log_probs >= 0.5).astype(int) #0.5 as baseline
log_probs[:10]
log_preds[:10]

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score, confusion_matrix

precision = precision_score(y_test, log_preds, zero_division=0)
recall = recall_score(y_test, log_preds, zero_division=0)
f1 = f1_score(y_test, log_preds, zero_division=0)
pr_auc = average_precision_score(y_test, log_probs)

print("Precision:", precision)
print("Recall:", recall)
print("F1 score:", f1)
print("PR-AUC:", pr_auc)

Precision: 0.06097560975609756
Recall: 0.9183673469387755
F1 score: 0.11435832274459974
PR-AUC: 0.7159122424484009


Confusion matrix

In [8]:
cm = confusion_matrix(y_test, log_preds)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Fraud"],
    columns=["Predicted Normal", "Predicted Fraud"]
)

cm_df

,Predicted Normal,Predicted Fraud
Actual Normal,55478,1386
Actual Fraud,8,90


ROC-AUC and Accuracy 

In [9]:
#Accuracy and ROC-AUC are included as reference metrics, but I focus more on precision, recall, and PR-AUC because the dataset is highly imbalanced.
from sklearn.metrics import accuracy_score, roc_auc_score

accuracy = accuracy_score(y_test, log_preds)
roc_auc = roc_auc_score(y_test, log_probs)

print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)

Accuracy: 0.9755275446789088
ROC-AUC: 0.9721687370080279


In [10]:
log_summary = pd.DataFrame({
    "model": ["Logistic Regression"],
    "precision": [precision],
    "recall": [recall],
    "f1": [f1],
    "pr_auc": [pr_auc],
    "accuracy_reference": [accuracy],
    "roc_auc_reference": [roc_auc]
})

log_summary.to_csv("../data/processed/logistic_regression_summary.csv", index=False)

log_summary

,model,precision,recall,f1,pr_auc,accuracy_reference,roc_auc_reference
0,Logistic Regression,0.060976,0.918367,0.114358,0.715912,0.975528,0.972169


### Random Forest Model

After the Logistic Regression baseline, I used Random Forest as a second model. Random Forest is useful here because fraud patterns may be non-linear and may depend on interactions between several features.

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [12]:
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = (rf_probs >= 0.5).astype(int)
rf_probs[:10]
rf_preds[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [13]:
rf_precision = precision_score(y_test, rf_preds, zero_division=0)
rf_recall = recall_score(y_test, rf_preds, zero_division=0)
rf_f1 = f1_score(y_test, rf_preds, zero_division=0)
rf_pr_auc = average_precision_score(y_test, rf_probs)
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_roc_auc = roc_auc_score(y_test, rf_probs)

print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("F1 score:", rf_f1)
print("PR-AUC:", rf_pr_auc)
print("Accuracy reference:", rf_accuracy)
print("ROC-AUC reference:", rf_roc_auc)

Precision: 0.6484375
Recall: 0.8469387755102041
F1 score: 0.7345132743362832
PR-AUC: 0.799998102786367
Accuracy reference: 0.9989466661985184
ROC-AUC reference: 0.9818797876494436


In [14]:
rf_cm = confusion_matrix(y_test, rf_preds)
rf_cm_df = pd.DataFrame(
    rf_cm,
    index=["Actual Normal", "Actual Fraud"],
    columns=["Predicted Normal", "Predicted Fraud"]
)
rf_cm_df

,Predicted Normal,Predicted Fraud
Actual Normal,56819,45
Actual Fraud,15,83


In [15]:
rf_summary = pd.DataFrame({
    "model": ["Random Forest"],
    "precision": [rf_precision],
    "recall": [rf_recall],
    "f1": [rf_f1],
    "pr_auc": [rf_pr_auc],
    "accuracy_reference": [rf_accuracy],
    "roc_auc_reference": [rf_roc_auc]
})
rf_summary.to_csv("../data/processed/random_forest_summary.csv", index=False)
rf_summary

,model,precision,recall,f1,pr_auc,accuracy_reference,roc_auc_reference
0,Random Forest,0.648438,0.846939,0.734513,0.799998,0.998947,0.98188


### Model Comparison

In [16]:
model_comparison = pd.concat([log_summary, rf_summary], ignore_index=True)
model_comparison.to_csv("../data/processed/model_comparison.csv", index=False)
model_comparison

,model,precision,recall,f1,pr_auc,accuracy_reference,roc_auc_reference
0,Logistic Regression,0.060976,0.918367,0.114358,0.715912,0.975528,0.972169
1,Random Forest,0.648438,0.846939,0.734513,0.799998,0.998947,0.981880


### Threshold Analysis

After comparing the two models, I used the Random Forest probability scores for threshold analysis because it had higher precision and PR-AUC than the Logistic Regression baseline.

In fraud detection, the default 0.5 threshold is not always the best choice. A lower threshold may catch more fraud cases but also creates more false positives and review workload. A higher threshold reduces false positives but may miss more fraud cases.

In [17]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90]
rows = []
for t in thresholds:
    preds = (rf_probs >= t).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    precision = precision_score(y_test, preds, zero_division=0)
    recall = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    review_workload = tp + fp
    rows.append({
        "threshold": t,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "review_workload": review_workload
    })
threshold_analysis = pd.DataFrame(rows)
threshold_analysis

,threshold,true_positives,false_positives,false_negatives,precision,recall,f1,review_workload
0,0.05,92,6915,6,0.013130,0.938776,0.025897,7007
1,0.10,91,2070,7,0.042110,0.928571,0.080567,2161
2,0.20,88,467,10,0.158559,0.897959,0.269525,555
3,0.30,87,184,11,0.321033,0.887755,0.471545,271
4,0.50,83,45,15,0.648438,0.846939,0.734513,128
5,0.70,76,15,22,0.835165,0.775510,0.804233,91
6,0.90,66,12,32,0.846154,0.673469,0.750000,78


In [18]:
threshold_analysis.to_csv("../data/processed/threshold_analysis.csv", index=False)

In [19]:
# cost analysis based on the same thresholds
test_result = X_test.copy()
test_result["actual_class"] = y_test.values
test_result["fraud_score"] = rf_probs
review_cost = 5
cost_list = []
for threshold in thresholds:
    test_result["predicted_class"] = 0
    test_result.loc[test_result["fraud_score"] >= threshold, "predicted_class"] = 1
    false_positive = test_result[
        (test_result["actual_class"] == 0) &
        (test_result["predicted_class"] == 1)
    ]
    false_negative = test_result[
        (test_result["actual_class"] == 1) &
        (test_result["predicted_class"] == 0)
    ]
    true_positive = test_result[
        (test_result["actual_class"] == 1) &
        (test_result["predicted_class"] == 1)
    ]
    false_positive_cost = len(false_positive) * review_cost
    missed_fraud_amount = false_negative["Amount"].sum()
    captured_fraud_amount = true_positive["Amount"].sum()
    total_estimated_cost = false_positive_cost + missed_fraud_amount
    cost_list.append([
        threshold,
        len(false_positive),
        len(false_negative),
        len(true_positive),
        false_positive_cost,
        missed_fraud_amount,
        captured_fraud_amount,
        total_estimated_cost
    ])
cost_analysis = pd.DataFrame(
    cost_list,
    columns=[
        "threshold",
        "false_positive_count",
        "missed_fraud_count",
        "captured_fraud_count",
        "false_positive_cost",
        "missed_fraud_amount",
        "captured_fraud_amount",
        "total_estimated_cost"
    ]
)
cost_analysis

,threshold,false_positive_count,missed_fraud_count,captured_fraud_count,false_positive_cost,missed_fraud_amount,captured_fraud_amount,total_estimated_cost
0,0.05,6915,6,92,34575,137.71,10507.22,34712.71
1,0.10,2070,7,91,10350,657.61,9987.32,11007.61
2,0.20,467,10,88,2335,1822.71,8822.22,4157.71
3,0.30,184,11,87,920,1826.50,8818.43,2746.50
4,0.50,45,15,83,225,3828.23,6816.70,4053.23
5,0.70,15,22,76,75,4421.44,6223.49,4496.44
6,0.90,12,32,66,60,5295.02,5349.91,5355.02


In [20]:
cost_analysis.to_csv("../data/processed/cost_analysis.csv", index=False)

In [21]:
scored_test = X_test.copy()
scored_test["actual_class"] = y_test.values
scored_test["fraud_score"] = rf_probs
# baseline threshold
scored_test["predicted_class_50"] = (scored_test["fraud_score"] >= 0.50).astype(int)
# selected threshold based on simple cost analysis
scored_test["predicted_class_30"] = (scored_test["fraud_score"] >= 0.30).astype(int)
scored_sample = scored_test.sample(1000, random_state=42)
scored_sample.to_csv("../data/sample/scored_transactions_sample.csv", index=False)
scored_sample.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V24,V25,V26,V27,V28,Amount,actual_class,fraud_score,predicted_class_50,predicted_class_30
2490,2061.0,-1.958955,-2.722373,0.065636,-2.051588,0.435947,4.509367,0.781314,0.993881,-0.800422,...,0.945037,0.535140,-0.381992,-0.126417,0.152883,624.03,0,0.039904,0,0
90487,63041.0,-2.438283,-0.264593,2.889100,3.375535,1.729515,-0.650652,-2.503850,-0.372552,-0.334218,...,0.569264,-0.323438,0.390778,0.651110,0.565652,33.37,0,0.026551,0,0
156589,108462.0,-1.318108,1.179453,-1.120372,0.206072,-1.600136,0.452594,2.032534,-3.290388,0.874368,...,0.595314,-1.075518,-0.259226,1.031443,0.026284,758.84,0,0.072681,0,0
138722,82811.0,-1.301314,1.216130,1.029738,1.292400,-0.062383,0.015550,0.404308,0.180961,0.073446,...,0.084947,-0.144056,-0.293500,-0.369949,-0.079365,20.23,0,0.026659,0,0
269869,163812.0,2.052545,-0.112153,-1.081547,0.416182,-0.177287,-1.146267,0.121181,-0.268025,0.649733,...,-0.033057,-0.338042,0.203471,-0.072333,-0.062021,1.98,0,0.009012,0,0


### Summary

Random Forest performed better than the Logistic Regression baseline in this experiment. Logistic Regression had high recall but very low precision, which means it caught many fraud cases but also created many false positives.

Random Forest had a better balance between precision and recall, and it also had a higher PR-AUC. Because of this, I used Random Forest fraud scores for the threshold and cost analysis.

The threshold analysis shows that a lower threshold catches more fraud but creates more false positives and review workload. A higher threshold reduces false positives but misses more fraud. Based on the simple cost assumption, threshold 0.30 had the lowest estimated cost in this experiment.